# dmipy-sim — load a `.ply` substrate, simulate, and visualise

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dmrai-lab/dmipy-sim/blob/main/examples/mesh_ply_and_viz.ipynb)

Run an arbitrary triangular surface mesh (closed **or** 3-D periodic) with the same
physics as the analytic geometries — restricted diffusion, surface relaxivity and
membrane permeability — plus the `dmipy_sim.viz` helpers for looking inside it.

The mesh here is generated on the fly so the notebook runs anywhere; swap in your own
file with `Mesh.from_ply("substrate.ply", scale=..., periodic=True, ...)`.


In [ ]:
%pip install -q "dmipy-sim[mesh]"
# (until released on PyPI, install from the repo:)
# %pip install -q "dmipy-sim[mesh] @ git+https://github.com/dmrai-lab/dmipy-sim.git"

## 1 · Build a mesh geometry

We tessellate a sphere with `trimesh` and wrap it in `Mesh`. `quality_report()` prints a
per-effect resolution verdict (permeability needs a finer surface than diffusion/relaxivity).


In [ ]:
import numpy as np, jax, jax.numpy as jnp, trimesh
import matplotlib.pyplot as plt
from dmipy_sim import Mesh, simulate, Sphere, set_b
from dmipy_sim.waveforms import Waveform

R, D = 5e-6, 2e-9                       # 5 µm cell, D = 2e-9 m^2/s
m = trimesh.creation.icosphere(subdivisions=4, radius=R)
mesh = Mesh(np.asarray(m.vertices), np.asarray(m.faces))
mesh.quality_report();

## 2 · A PGSE waveform, and the mesh signal vs the analytic sphere

Same waveform/seed/walkers through the mesh and the analytic `Sphere` — the mesh matches
to the Monte-Carlo noise floor.


In [ ]:
def pgse(nb=6, n_t=300, TE=40e-3):
    dt = TE / (n_t - 1); G = np.zeros((1, n_t, 3), np.float32)
    G[0, 1:int(0.4*n_t), 0] = 1.0; G[0, -int(0.4*n_t):-1, 0] = -1.0
    return set_b(Waveform(G=jnp.tile(jnp.array(G), (nb,1,1)), dt=dt, echo_idx=n_t-1),
                 np.linspace(1, 2.5e9, nb))
wf = pgse()
s_mesh = np.asarray(simulate(20_000, D, wf, mesh, seed=0))
s_ana  = np.asarray(simulate(20_000, D, wf, Sphere(radius=R), seed=0))
print("max |mesh - analytic Sphere| =", np.abs(s_mesh - s_ana).max())

## 3 · Surface relaxivity and permeability

Both are substrate properties set on the geometry (one walk each).


In [ ]:
s_rho  = np.asarray(simulate(20_000, D, wf, Mesh(np.asarray(m.vertices), np.asarray(m.faces),
                                                 surface_relaxivity_t2=5e-6), seed=0))
s_kap  = np.asarray(simulate(20_000, D, wf, Mesh(np.asarray(m.vertices), np.asarray(m.faces),
                                                 permeability=2e-5), seed=0))
plt.plot(s_mesh, "o-", label="diffusion"); plt.plot(s_rho, "s-", label="+ surface relaxivity")
plt.plot(s_kap, "^-", label="+ permeability"); plt.legend(); plt.xlabel("b index"); plt.ylabel("signal"); plt.show()

## 4 · Trajectory export — pick the walkers that permeated

`return_positions='full'` gives per-timestep positions; with `return_compartments='full'`
you can select walkers that crossed the membrane and visualise only their paths.


In [ ]:
_, pos, origin, comp = simulate(2_000, D, pgse(1, 150), Mesh(np.asarray(m.vertices),
        np.asarray(m.faces), permeability=2e-5), seed=0,
        return_positions="full", return_compartments="full")
permeated = (comp != comp[:, :1]).any(axis=1)
print(f"{permeated.sum()} / {len(permeated)} walkers permeated; positions {pos.shape}")

## 5 · Look inside — the `dmipy_sim.viz` helpers

A plane cross-section with walkers, and a true-3D view of walker paths confined inside a
transparent cell (`save_rotation` writes an animated GIF).


In [ ]:
from dmipy_sim import (plot_mesh_section, plot_walkers_3d, plot_mesh_3d,
                       seed_in_cell, walk_paths, plot_trajectories)
from dmipy_sim.viz import _split_cells

walkers = np.asarray(mesh.init_positions(3000, jax.random.PRNGKey(0), intra=True))
plot_mesh_section(mesh, axis="z", offset=0.0, walkers=walkers); plt.show()

cell   = _split_cells(mesh)[0]
paths  = walk_paths(mesh, 12, 400, diffusivity=D, dt=2e-4, r0=seed_in_cell(cell, 12))
ax = plot_mesh_3d(mesh, cells=(0,), paths=paths); plt.show()
# ax = ...; save_rotation(ax, "cell_spin.gif")   # animated GIF for slides

## Your own substrate

```python
mesh = Mesh.from_ply('substrate.ply', scale=1e-5,      # normalised coords -> metres
                     periodic=True, voxel_min=[-10e-6]*3, voxel_max=[10e-6]*3,
                     feature_radius=1.7e-6, permeability=2e-5)
```

Pass `orientation=` (or `R=`) to place the substrate in the bore relative to B0 = +z.
See the rendered gallery in [`examples/mesh_viz/`](https://github.com/dmrai-lab/dmipy-sim/tree/main/examples/mesh_viz).
